# Pytorch Lightning

In [1]:
import pytorch_lightning as pl
import torch
import torch.nn as nn

from torchmetrics import Accuracy

## Модель как LightningModule

Lightning объединяет модель и цикл обучения в одном классе. Метрики — torchmetrics `Accuracy` (`task='multiclass'`): накапливаются по батчам, в конце эпохи логируются и сбрасываются.

In [2]:
class MultiLayerPerceptron(pl.LightningModule):
    def __init__(self, image_shape=(1, 28, 28), hidden_units=(32, 16)):
        super().__init__()
        self.train_acc = Accuracy(task='multiclass', num_classes=10)
        self.valid_acc = Accuracy(task='multiclass', num_classes=10)
        self.test_acc = Accuracy(task='multiclass', num_classes=10) 

        input_size = image_shape[0] * image_shape[1] * image_shape[2]
        all_layers = [nn.Flatten()]
        for hidden_unit in hidden_units:
            layer = nn.Linear(input_size, hidden_unit)
            all_layers.append(layer)
            all_layers.append(nn.ReLU())
            input_size = hidden_unit 

        all_layers.append(nn.Linear(hidden_units[-1], 10))
        self.model = nn.Sequential(*all_layers)

    def forward(self, x):
        x = self.model(x) 
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(self(x), y)
        preds = torch.argmax(logits, dim=1)
        self.train_acc.update(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self.log("train_acc", self.train_acc.compute())
        self.train_acc.reset()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(self(x), y)
        preds = torch.argmax(logits, dim=1)
        self.valid_acc.update(preds, y)
        self.log("valid_loss", loss, prog_bar=True)
        self.log("valid_acc", self.valid_acc.compute(), prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, у = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(self(x), у)
        preds = torch.argmax(logits, dim=1)
        self.test_acc.update(preds, у)
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", self.test_acc.compute(), prog_bar=True)
        return loss 

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        return optimizer

In [3]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torchvision.datasets import MNIST
from torchvision import transforms 

## Данные как LightningDataModule

`prepare_data` — скачивание один раз; `setup` — сплит train (55000) / val (5000) и тест; даталоадеры по 64. Все этапы подготовки данных инкапсулированы в датамодуль.

In [4]:
class MnistDataModule(pl.LightningDataModule):
    def __init__(self, data_path='./'):
        super().__init__()
        self.data_path = data_path
        self.transform = transforms.Compose([transforms.ToTensor()])

    def prepare_data(self):
        MNIST(root=self.data_path,download=True)

    def setup(self, stage=None):
        # stage может принимать значения
        # 'fit', 'validate', 'test', or 'predict'
        # здесь укажите нужное
        mnist_all = MNIST(
            root=self.data_path,
            train=True,
            transform=self.transform,
            download=False
        )

        self.train, self.val = random_split(
            mnist_all, [55000, 5000], generator=torch.Generator().manual_seed(1) 
        )

        self.test = MNIST(
            root=self.data_path,
            train=False,
            transform=self.transform,
            download=False 
        )

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=64, num_workers=0) 

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=64, num_workers=0)

    def test_dataloader(self):
        return DataLoader(self.test, batch_size=64, num_workers=0) 

        

In [5]:
torch.manual_seed(1)
mnist_dm = MnistDataModule() 

## Обучение: Trainer

`pl.Trainer` управляет всем процессом: `fit(model, datamodule)` сам перебирает эпохи и запускает train/val шаги.

In [7]:
mnistclassifier = MultiLayerPerceptron()

trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto',
    devices=1
)

trainer.fit(model=mnistclassifier, datamodule=mnist_dm) 

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name      | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------
0 | train_acc | MulticlassAccuracy | 0      | train | 0    
1 | valid_acc | MulticlassAccuracy | 0      | train | 0    
2 | test_acc  | MulticlassAccuracy | 0      | train | 0    
3 | model     | Sequential         | 25.8 K | train | 0    
-----------------------------------------------------------------
25.8 K    Trainable params
0         Non-trainable params

d:\My\Study\programming\pytorch-learn\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
d:\My\Study\programming\pytorch-learn\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
d:\My\Study\programming\pytorch-learn\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 9: 100%|██████████| 860/860 [00:28<00:00, 30.26it/s, v_num=4, train_loss=0.0912, valid_loss=0.161, valid_acc=0.940] 

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 860/860 [00:28<00:00, 30.23it/s, v_num=4, train_loss=0.0912, valid_loss=0.161, valid_acc=0.940]
